# 🎯 Adım 4 — MRMR Özellik Seçimi

Ön işleme sonrası elimizde **147 özellik** kaldı.  
Hepsi birden modele girerse iki sorun çıkar:
- Model "gereksiz" özelliklerden gürültü öğrenir
- 69 hasta için 147 özellik fazla → **overfitting** riski

**MRMR** bu sorunu çözer: hem bilgi taşıyan hem de birbirinden bağımsız özellikleri seçer.

> **Finansal analoji:** Bir portföy yöneticisinin hisse seçimi gibi düşün.  
> Sadece getirisi yüksek hisseleri seçmek yetmez — birbirleriyle yüksek korelasyonlu hisseler  
> portföye çeşitlendirme katkısı yapmaz. MRMR da aynı şeyi yapar:
> **Relevance** = getiri (sınıfı ne kadar iyi açıklıyor?)  
> **Redundancy** = korelasyon (diğer seçilmiş özelliklerle ne kadar örtüşüyor?)

---

In [ ]:
# Önce kütüphaneyi kur (bir kez çalıştır)
import subprocess
subprocess.run(['pip', 'install', 'mrmr-selection', '-q'], check=True)
print('mrmr-selection kuruldu ✓')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mrmr import mrmr_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
print('Kütüphaneler hazır ✓')

In [ ]:
# 03 notebook'undan pipeline ve splits'i yeniden oluştur
# (Notebook'lar bağımsız çalışabilsin diye tüm setup burada)

normal = pd.read_csv('../data/normal_radiomics.csv')
papil  = pd.read_csv('../data/papilodem_radiomics.csv')
normal['label'] = 0
papil['label']  = 1
df = pd.concat([normal, papil], ignore_index=True)
feature_cols = [c for c in df.columns if c.startswith('Feature_')]

def hasta_bazinda_bol(df, test_ratio=0.20, val_ratio=0.10, random_state=42):
    rng = np.random.RandomState(random_state)
    hasta_etiket = df.groupby('PatientIndex')['label'].first()
    train_idx, val_idx, test_idx = [], [], []
    for sinif in [0, 1]:
        hastalar = hasta_etiket[hasta_etiket == sinif].index.tolist()
        rng.shuffle(hastalar)
        n = len(hastalar)
        n_test = max(1, int(n * test_ratio))
        n_val  = max(1, int(n * val_ratio))
        test_idx  += hastalar[:n_test]
        val_idx   += hastalar[n_test:n_test + n_val]
        train_idx += hastalar[n_test + n_val:]
    return (df[df['PatientIndex'].isin(train_idx)].copy(),
            df[df['PatientIndex'].isin(val_idx)].copy(),
            df[df['PatientIndex'].isin(test_idx)].copy())

class RadyomikOnIsleme:
    def __init__(self, variance_threshold=0.01, correlation_threshold=0.95):
        self.variance_threshold    = variance_threshold
        self.correlation_threshold = correlation_threshold
        self.imputer      = SimpleImputer(strategy='median')
        self.var_selector = VarianceThreshold(threshold=variance_threshold)
        self.scaler       = RobustScaler()
        self.selected_features_after_var  = None
        self.selected_features_after_corr = None
        self.high_corr_to_drop = None

    def fit(self, X_train, feature_names):
        X_train = np.where(np.isinf(X_train), np.nan, X_train)
        X = self.imputer.fit_transform(X_train)
        self.var_selector.fit(X)
        mask_var = self.var_selector.get_support()
        self.selected_features_after_var = [f for f, m in zip(feature_names, mask_var) if m]
        X = X[:, mask_var]
        corr_df = pd.DataFrame(X, columns=self.selected_features_after_var)
        upper = corr_df.corr().abs().where(
            np.triu(np.ones((len(self.selected_features_after_var),)*2), k=1).astype(bool))
        self.high_corr_to_drop = [col for col in upper.columns if any(upper[col] > self.correlation_threshold)]
        self.selected_features_after_corr = [f for f in self.selected_features_after_var
                                              if f not in self.high_corr_to_drop]
        keep_idx = [self.selected_features_after_var.index(f) for f in self.selected_features_after_corr]
        self.scaler.fit(X[:, keep_idx])
        return self

    def transform(self, X, feature_names):
        X = np.where(np.isinf(X), np.nan, X)
        X = self.imputer.transform(X)
        X = pd.DataFrame(X, columns=feature_names)[self.selected_features_after_var].values
        X = pd.DataFrame(X, columns=self.selected_features_after_var)[self.selected_features_after_corr].values
        return self.scaler.transform(X)

    def fit_transform(self, X_train, feature_names):
        self.fit(X_train, feature_names)
        return self.transform(X_train, feature_names)

# Split 0 hazırla
train_df, val_df, test_df = hasta_bazinda_bol(df, random_state=42)
pipeline = RadyomikOnIsleme()
X_train_proc = pipeline.fit_transform(train_df[feature_cols].values, feature_cols)
X_val_proc   = pipeline.transform(val_df[feature_cols].values, feature_cols)
y_train = train_df['label'].values
y_val   = val_df['label'].values

proc_cols = pipeline.selected_features_after_corr
print(f'Ön işleme sonrası: {len(proc_cols)} özellik')
print('Setup hazır ✓')

---
## MRMR Nasıl Çalışır?

MRMR her adımda şu soruyu sorar:  
"Henüz seçilmemiş özellikler arasında, **hedefe en fazla bilgi katan** ve **zaten seçilmiş özelliklerle en az örtüşen** hangisi?"

**MRMR skoru:**
$$\text{MRMR}(f) = \underbrace{I(f; y)}_{\text{Relevance}} - \underbrace{\frac{1}{|S|} \sum_{s \in S} |r(f, s)|}_{\text{Redundancy}}$$

- **Relevance:** Mutual Information — özellik ile hedef arasındaki bilgi paylaşımı  
- **Redundancy:** Seçilmiş özelliklerle ortalama Pearson korelasyonu  
- Her turda en yüksek skorlu özellik seçilir ve listeye eklenir

In [ ]:
# MRMR'ı çalıştır — k=30 en iyi özellik
# Sadece train verisine uygulanır!
print('MRMR hesaplanıyor (biraz sürebilir)...')

K_MAX = 30
train_proc_df = pd.DataFrame(X_train_proc, columns=proc_cols)

selected_features = mrmr_classif(
    X=train_proc_df,
    y=pd.Series(y_train),
    K=K_MAX
)

print(f'\nMRMR tamamlandı ✓')
print(f'Seçilen {K_MAX} özellik (sıralı önem):')
for i, feat in enumerate(selected_features, 1):
    print(f'  {i:2d}. {feat}')

---
## Kaç özellik seçmeliyiz? (k optimizasyonu)

MRMR kaç özellik seçeceğimizi söylemez — biz karar vermeliyiz.  
Farklı k değerleri için val seti F1 skorunu ölçelim.  
En yüksek F1 veren k'yı kullanırız.

In [ ]:
k_degerler = [5, 8, 10, 15, 20, 25, 30]
sonuclar = []

for k in k_degerler:
    feat_k = selected_features[:k]
    
    X_tr_k = train_proc_df[feat_k].values
    X_va_k = pd.DataFrame(X_val_proc, columns=proc_cols)[feat_k].values
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_tr_k, y_train)
    
    y_pred = rf.predict(X_va_k)
    f1 = f1_score(y_val, y_pred, average='macro')
    sonuclar.append({'k': k, 'val_macro_f1': round(f1, 4)})
    print(f'  k={k:2d} → Val Macro-F1: {f1:.4f}')

sonuc_df = pd.DataFrame(sonuclar)
best_k = sonuc_df.loc[sonuc_df['val_macro_f1'].idxmax(), 'k']
print(f'\n✓ En iyi k = {best_k} (Val Macro-F1: {sonuc_df["val_macro_f1"].max():.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: k vs F1
axes[0].plot(sonuc_df['k'], sonuc_df['val_macro_f1'],
             marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'En iyi k={best_k}')
axes[0].set_xlabel('Seçilen özellik sayısı (k)')
axes[0].set_ylabel('Val Macro-F1')
axes[0].set_title('MRMR k Optimizasyonu')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sağ: En iyi k özelliklerinin RF feature importance sıralaması
best_feats = selected_features[:best_k]
X_best = train_proc_df[best_feats].values
rf_best = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_best.fit(X_best, y_train)

imp_df = pd.DataFrame({'feature': best_feats,
                        'importance': rf_best.feature_importances_})\
           .sort_values('importance', ascending=True)

axes[1].barh(imp_df['feature'], imp_df['importance'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('RF Feature Importance')
axes[1].set_title(f'Seçilen {best_k} Özelliğin Önemi (RF)')
axes[1].tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('../figures/fig_mrmr_k_optimizasyon.png', dpi=150, bbox_inches='tight')
plt.show()

---
## MRMR vs Sadece Relevance (farkı görelim)

MRMR'ın neden daha iyi olduğunu anlamak için karşılaştırma:
- **Sadece Mutual Information (MI):** En yüksek MI skorlu özellikleri seç  
- **MRMR:** MI yüksek VE birbirinden bağımsız özellikleri seç

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Sadece MI ile seç
mi_scores = mutual_info_classif(X_train_proc, y_train, random_state=42)
mi_df = pd.DataFrame({'feature': proc_cols, 'mi': mi_scores})\
          .sort_values('mi', ascending=False)
mi_only_feats = mi_df.head(best_k)['feature'].tolist()

# MI-only ile F1
X_mi_tr = train_proc_df[mi_only_feats].values
X_mi_va = pd.DataFrame(X_val_proc, columns=proc_cols)[mi_only_feats].values
rf_mi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_mi.fit(X_mi_tr, y_train)
f1_mi = f1_score(y_val, rf_mi.predict(X_mi_va), average='macro')

# MRMR ile F1
X_mrmr_va = pd.DataFrame(X_val_proc, columns=proc_cols)[best_feats].values
f1_mrmr = f1_score(y_val, rf_best.predict(X_mrmr_va), average='macro')

print(f'k = {best_k} özellik ile karşılaştırma:')
print(f'  Sadece MI  → Val Macro-F1: {f1_mi:.4f}')
print(f'  MRMR       → Val Macro-F1: {f1_mrmr:.4f}')
fark = f1_mrmr - f1_mi
yon = 'daha iyi ✓' if fark > 0 else 'daha düşük'
print(f'  MRMR {abs(fark)*100:.2f} puan {yon}')

---
## Özet

In [ ]:
print('=' * 55)
print('MRMR ÖZELLİK SEÇİMİ ÖZET')
print('=' * 55)
print(f'Ön işleme sonrası özellik sayısı : {len(proc_cols)}')
print(f'MRMR k_max                       : {K_MAX}')
print(f'Optimal k (val F1 bazlı)         : {best_k}')
print(f'Val Macro-F1 (k={best_k})          : {sonuc_df[sonuc_df.k==best_k]["val_macro_f1"].values[0]:.4f}')
print(f'Seçilen özellikler               : {best_feats}')
print('=' * 55)
print('\nBu k değeri ve özellikler bir sonraki notebook\'ta')
print('Optuna hiperparametre optimizasyonunda kullanılacak.')
print('\nSıradaki adım → 05_model_egitimi_optuna.ipynb')

# Diğer notebook'larda kullanmak için kaydet
pd.Series(best_feats).to_csv('../results/mrmr_selected_features.csv', index=False, header=False)
print(f'\nSeçilen özellikler kaydedildi: results/mrmr_selected_features.csv')